# 15 — Testing, Linting, and Code Quality

Goal: ship correct code reliably using tests and automated quality gates.

_Generated: 2026-02-19_

## Setup

This course targets **Python 3.11+** (works on 3.10+, with a few feature differences).

Recommended tooling:

```bash
# create + activate a virtual environment
python -m venv .venv
# mac/linux:
source .venv/bin/activate
# windows (PowerShell):
# .venv\Scripts\Activate.ps1

python -m pip install -U pip

# quality-of-life (optional but recommended)
python -m pip install -U ipykernel ruff black pytest mypy
python -m pip install -U pytest pytest-cov hypothesis
```

If you're using Jupyter:
```bash
python -m ipykernel install --user --name python-course --display-name "Python Course (.venv)"
```

In [ ]:

import sys, platform, os
print("python:", sys.version.split()[0])
print("implementation:", platform.python_implementation())
print("platform:", platform.platform())
print("cwd:", os.getcwd())


## 1.
L1: Why testing matters

Testing answers:
- “Did I break something?”
- “Does this edge case behave?”
- “Can I refactor confidently?”

Core categories:
- unit tests
- integration tests
- end-to-end tests

## 2.
L2: `unittest` (stdlib)

Good for: environments where you can’t install pytest.

In [ ]:

import unittest

def add(a: int, b: int) -> int:
    return a + b

class TestAdd(unittest.TestCase):
    def test_add(self):
        self.assertEqual(add(2, 3), 5)

# In a script you'd run:
# if __name__ == "__main__":
#     unittest.main()
print("Defined unittest test (run with unittest.main() in a script).")


## 3.
L3: `pytest` (modern default)

Pytest shines because:
- plain `assert`
- fixtures
- parametrization

Example file `tests/test_math.py`:

```python
import pytest

@pytest.mark.parametrize("a,b,expected", [(2,3,5), (0,0,0)])
def test_add(a,b,expected):
    assert a+b == expected
```

Run:
```bash
pytest -q
pytest --cov=src
```

## 4.
L4: Mocking (control dependencies)

Mocking is useful when:
- you call external services
- you depend on time/randomness
- you want to isolate side effects

Prefer dependency injection + small pure functions.

In [ ]:

from unittest.mock import Mock

def fetch_user(api_get, user_id: int):
    return api_get(f"/users/{user_id}")

api_get = Mock(return_value={"id": 1, "name": "Ada"})
u = fetch_user(api_get, 1)

api_get.assert_called_once_with("/users/1")
print(u)


## 5.
L5: Linting + formatting + type checking

A typical pipeline:

```bash
ruff check .
black .
mypy src
pytest
```

Automate with GitHub Actions and/or `pre-commit`.

## 6.
L6: Exercises

1. Write tests for a `clamp()` function using pytest parametrization.
2. Mock a function that reads from the network and test your parsing.
3. Add a type-checked function and run mypy/pyright.

## 7.
L7: Doctest (tests in docstrings)

Doctest runs examples embedded in docstrings.
Good for small pure functions and documentation accuracy.

In [ ]:

def inc(x: int) -> int:
    """Increment x.

    >>> inc(1)
    2
    >>> inc(-1)
    0
    """
    return x + 1

import doctest
doctest.testmod(verbose=False)
print("doctest ok")


## 8.
L8: Property-based testing (Hypothesis)

Instead of writing individual examples, define properties that must always hold.
(This requires installing `hypothesis`.)

In [ ]:

try:
    from hypothesis import given, strategies as st

    @given(st.lists(st.integers()))
    def test_reverse_twice(xs):
        assert list(reversed(list(reversed(xs)))) == xs

    test_reverse_twice()
    print("hypothesis ran")
except Exception as e:
    print("hypothesis not available:", e)
